In [1]:
import pandas as pd

# --- Configuration ---
input_csv_dir = './data/'

# --- Read CSV file ---
# Read the CSV file with specified columns
columns = ['CMPLNT_NUM', 'CMPLNT_FR_DT', 'CMPLNT_FR_TM', 'KY_CD', 'OFNS_DESC',
           'LAW_CAT_CD', 'BORO_NM', 'PREM_TYP_DESC', 'Latitude', 'Longitude']
df1 = pd.read_csv(input_csv_dir + 'NYPD_Complaint_Data_Historic.csv', usecols=columns)

# Combine date and time columns into a datetime object (handling invalid entries as NaT)
df1['CMPLNT_DTTM'] = pd.to_datetime(
    df1['CMPLNT_FR_DT'] + ' ' + df1['CMPLNT_FR_TM'],
    errors='coerce'
)

# Fill Missing values in PREM_TYP_DESC with 'UNKNOWN'
df1['PREM_TYP_DESC'] = df1['PREM_TYP_DESC'].fillna('UNKNOWN')

# Drop the original date and time columns
df1 = df1.drop(columns=['CMPLNT_FR_DT', 'CMPLNT_FR_TM'])

# Drop duplicates
df1 = df1.drop_duplicates()

In [2]:
df1

,CMPLNT_NUM,KY_CD,OFNS_DESC,LAW_CAT_CD,BORO_NM,PREM_TYP_DESC,Latitude,Longitude,CMPLNT_DTTM
0,101109527,113,FORGERY,FELONY,BRONX,BAR/NIGHT CLUB,40.828848,-73.916661,2015-12-31 23:45:00
1,153401121,101,MURDER & NON-NEGL. MANSLAUGHTER,FELONY,QUEENS,UNKNOWN,40.697338,-73.784557,2015-12-31 23:36:00
2,569369778,117,DANGEROUS DRUGS,FELONY,MANHATTAN,OTHER,40.802607,-73.945052,2015-12-31 23:30:00
3,968417082,344,ASSAULT 3 & RELATED OFFENSES,MISDEMEANOR,QUEENS,RESIDENCE-HOUSE,40.654549,-73.726339,2015-12-31 23:30:00
4,641637920,344,ASSAULT 3 & RELATED OFFENSES,MISDEMEANOR,MANHATTAN,OTHER,40.738002,-73.987891,2015-12-31 23:25:00
...,...,...,...,...,...,...,...,...,...
1048570,317154175,341,PETIT LARCENY,MISDEMEANOR,BRONX,RESIDENCE - APT. HOUSE,40.806932,-73.920767,2013-11-01 20:30:00
1048571,833410354,344,ASSAULT 3 & RELATED OFFENSES,MISDEMEANOR,BROOKLYN,GROCERY/BODEGA,40.660901,-73.895227,2013-11-01 20:28:00
1048572,961949188,121,CRIMINAL MISCHIEF & RELATED OF,FELONY,MANHATTAN,FAST FOOD,40.723909,-74.004681,2013-11-01 20:25:00
1048573,157608118,235,DANGEROUS DRUGS,MISDEMEANOR,BRONX,STREET,40.877554,-73.872939,2013-11-01 20:25:00


In [3]:
datetime_dim = df1[['CMPLNT_DTTM']].drop_duplicates().reset_index(drop=True)
datetime_dim['year'] = datetime_dim['CMPLNT_DTTM'].dt.year.fillna(-1).astype(int)
datetime_dim['month'] = datetime_dim['CMPLNT_DTTM'].dt.month.fillna(-1).astype(int)
datetime_dim['day'] = datetime_dim['CMPLNT_DTTM'].dt.day.fillna(-1).astype(int)
datetime_dim['weekday'] = datetime_dim['CMPLNT_DTTM'].dt.weekday.fillna(-1).astype(int)
datetime_dim['hour'] = datetime_dim['CMPLNT_DTTM'].dt.hour.fillna(-1).astype(int)
datetime_dim['minute'] = datetime_dim['CMPLNT_DTTM'].dt.minute.fillna(-1).astype(int)
datetime_dim['datetime_id'] = datetime_dim.index
datetime_dim = datetime_dim[['datetime_id', 'CMPLNT_DTTM', 'year', 'month', 'day', 'weekday', 'hour', 'minute']]

In [4]:
datetime_dim

,datetime_id,CMPLNT_DTTM,year,month,day,weekday,hour,minute
0,0,2015-12-31 23:45:00,2015,12,31,3,23,45
1,1,2015-12-31 23:36:00,2015,12,31,3,23,36
2,2,2015-12-31 23:30:00,2015,12,31,3,23,30
3,3,2015-12-31 23:25:00,2015,12,31,3,23,25
4,4,2015-12-31 23:18:00,2015,12,31,3,23,18
...,...,...,...,...,...,...,...,...
295050,295050,2013-11-01 20:55:00,2013,11,1,4,20,55
295051,295051,2013-11-01 20:45:00,2013,11,1,4,20,45
295052,295052,2013-11-01 20:35:00,2013,11,1,4,20,35
295053,295053,2013-11-01 20:28:00,2013,11,1,4,20,28


In [5]:
offense_dim = df1[['KY_CD', 'OFNS_DESC']].drop_duplicates().reset_index(drop=True)
offense_dim['offense_id'] = offense_dim.index
offense_dim = offense_dim[['offense_id', 'KY_CD', 'OFNS_DESC']]

In [6]:
offense_dim

,offense_id,KY_CD,OFNS_DESC
0,0,113,FORGERY
1,1,101,MURDER & NON-NEGL. MANSLAUGHTER
2,2,117,DANGEROUS DRUGS
3,3,344,ASSAULT 3 & RELATED OFFENSES
4,4,106,FELONY ASSAULT
...,...,...,...
82,82,577,UNDER THE INFLUENCE OF DRUGS
83,83,881,OTHER TRAFFIC INFRACTION
84,84,103,"HOMICIDE-NEGLIGENT,UNCLASSIFIE"
85,85,360,LOITERING FOR DRUG PURPOSES


In [7]:
law_cat_dim = df1[['LAW_CAT_CD']].drop_duplicates().reset_index(drop=True)
law_cat_dim['law_cat_id'] = law_cat_dim.index
law_cat_dim = law_cat_dim[['law_cat_id', 'LAW_CAT_CD']]

In [8]:
law_cat_dim

,law_cat_id,LAW_CAT_CD
0,0,FELONY
1,1,MISDEMEANOR
2,2,VIOLATION


In [9]:
premise_dim = df1[['PREM_TYP_DESC']].drop_duplicates().reset_index(drop=True)
premise_dim['premise_id'] = premise_dim.index
premise_dim = premise_dim[['premise_id', 'PREM_TYP_DESC']]

In [10]:
premise_dim

,premise_id,PREM_TYP_DESC
0,0,BAR/NIGHT CLUB
1,1,UNKNOWN
2,2,OTHER
3,3,RESIDENCE-HOUSE
4,4,DRUG STORE
...,...,...
66,66,PHOTO/COPY
67,67,MOSQUE
68,68,LOAN COMPANY
69,69,CEMETERY


In [11]:
# Borough population
df2 = pd.read_csv(input_csv_dir + 'Population_by_Borough_NYC.csv', usecols=['Borough', '2010']).rename(columns={'Borough': 'BORO_NM', '2010': 'population'})
df2['BORO_NM'] = df2['BORO_NM'].str.upper().str.strip()
df2 = df2[df2['BORO_NM'] != 'NYC TOTAL']
df2['population'] = df2['population'].str.replace(',', '').astype(int)

borough_dim = df1[['BORO_NM']].drop_duplicates().reset_index(drop=True)
borough_dim['borough_id'] = borough_dim.index
borough_dim = borough_dim[['borough_id', 'BORO_NM']]
borough_dim = borough_dim.merge(df2[['BORO_NM', 'population']], on='BORO_NM', how='left')

In [12]:
borough_dim

,borough_id,BORO_NM,population
0,0,BRONX,1385108
1,1,QUEENS,2250002
2,2,MANHATTAN,1585873
3,3,BROOKLYN,2552911
4,4,STATEN ISLAND,468730


In [13]:
# --- Construcción de la tabla crime_fact ---

# Paso 1: Seleccionar columnas clave de df1
fact_df = df1[[
    'CMPLNT_NUM', 'CMPLNT_DTTM', 'KY_CD', 'OFNS_DESC',
    'LAW_CAT_CD', 'PREM_TYP_DESC', 'BORO_NM',
    'Latitude', 'Longitude'
]].copy()

# Paso 2: Merge con datetime_dim para obtener datetime_id
fact_df = fact_df.merge(
    datetime_dim[['datetime_id', 'CMPLNT_DTTM']],
    on='CMPLNT_DTTM', how='left'
)

# Paso 3: Merge con offense_dim para obtener offense_id
fact_df = fact_df.merge(
    offense_dim[['offense_id', 'KY_CD', 'OFNS_DESC']],
    on=['KY_CD', 'OFNS_DESC'], how='left'
)

# Paso 4: Merge con law_cat_dim para obtener law_cat_id
fact_df = fact_df.merge(
    law_cat_dim[['law_cat_id', 'LAW_CAT_CD']],
    on='LAW_CAT_CD', how='left'
)

# Paso 5: Merge con premise_dim para obtener premise_id
fact_df = fact_df.merge(
    premise_dim[['premise_id', 'PREM_TYP_DESC']],
    on='PREM_TYP_DESC', how='left'
)

# Paso 6: Merge con borough_dim para obtener borough_id
fact_df = fact_df.merge(
    borough_dim[['borough_id', 'BORO_NM']],
    on='BORO_NM', how='left'
)

# Paso 7: Seleccionar y renombrar columnas al esquema final
crime_fact = fact_df[[
    'CMPLNT_NUM', 'datetime_id', 'offense_id',
    'law_cat_id', 'premise_id', 'borough_id',
    'Latitude', 'Longitude'
]].rename(columns={
    'Latitude': 'LATITUDE',
    'Longitude': 'LONGITUDE'
})

# Paso 8: Limpiar el índice
crime_fact = crime_fact.reset_index(drop=True)

In [14]:
crime_fact

,CMPLNT_NUM,datetime_id,offense_id,law_cat_id,premise_id,borough_id,LATITUDE,LONGITUDE
0,101109527,0,0,0,0,0,40.828848,-73.916661
1,153401121,1,1,0,1,1,40.697338,-73.784557
2,569369778,2,2,0,2,2,40.802607,-73.945052
3,968417082,2,3,1,3,1,40.654549,-73.726339
4,641637920,3,3,1,2,2,40.738002,-73.987891
...,...,...,...,...,...,...,...,...
1048570,317154175,227659,7,1,6,0,40.806932,-73.920767
1048571,833410354,295053,3,1,21,3,40.660901,-73.895227
1048572,961949188,295054,26,0,7,2,40.723909,-74.004681
1048573,157608118,295054,5,1,5,0,40.877554,-73.872939


In [17]:
reconstructed_df = crime_fact \
    .merge(datetime_dim[['datetime_id', 'CMPLNT_DTTM']], on='datetime_id', how='left') \
    .merge(offense_dim[['offense_id', 'KY_CD', 'OFNS_DESC']], on='offense_id', how='left') \
    .merge(law_cat_dim[['law_cat_id', 'LAW_CAT_CD']], on='law_cat_id', how='left') \
    .merge(premise_dim[['premise_id', 'PREM_TYP_DESC']], on='premise_id', how='left') \
    .merge(borough_dim[['borough_id', 'BORO_NM']], on='borough_id', how='left')

# Orden opcional para que se parezca al original
reconstructed_df = reconstructed_df[[
    'CMPLNT_NUM', 'CMPLNT_DTTM', 'KY_CD', 'OFNS_DESC',
    'LAW_CAT_CD', 'PREM_TYP_DESC', 'BORO_NM', 'LATITUDE', 'LONGITUDE'
]]

In [18]:
reconstructed_df

,CMPLNT_NUM,CMPLNT_DTTM,KY_CD,OFNS_DESC,LAW_CAT_CD,PREM_TYP_DESC,BORO_NM,LATITUDE,LONGITUDE
0,101109527,2015-12-31 23:45:00,113,FORGERY,FELONY,BAR/NIGHT CLUB,BRONX,40.828848,-73.916661
1,153401121,2015-12-31 23:36:00,101,MURDER & NON-NEGL. MANSLAUGHTER,FELONY,UNKNOWN,QUEENS,40.697338,-73.784557
2,569369778,2015-12-31 23:30:00,117,DANGEROUS DRUGS,FELONY,OTHER,MANHATTAN,40.802607,-73.945052
3,968417082,2015-12-31 23:30:00,344,ASSAULT 3 & RELATED OFFENSES,MISDEMEANOR,RESIDENCE-HOUSE,QUEENS,40.654549,-73.726339
4,641637920,2015-12-31 23:25:00,344,ASSAULT 3 & RELATED OFFENSES,MISDEMEANOR,OTHER,MANHATTAN,40.738002,-73.987891
...,...,...,...,...,...,...,...,...,...
1048570,317154175,2013-11-01 20:30:00,341,PETIT LARCENY,MISDEMEANOR,RESIDENCE - APT. HOUSE,BRONX,40.806932,-73.920767
1048571,833410354,2013-11-01 20:28:00,344,ASSAULT 3 & RELATED OFFENSES,MISDEMEANOR,GROCERY/BODEGA,BROOKLYN,40.660901,-73.895227
1048572,961949188,2013-11-01 20:25:00,121,CRIMINAL MISCHIEF & RELATED OF,FELONY,FAST FOOD,MANHATTAN,40.723909,-74.004681
1048573,157608118,2013-11-01 20:25:00,235,DANGEROUS DRUGS,MISDEMEANOR,STREET,BRONX,40.877554,-73.872939
